# Phase 1 — Data + Environment

This notebook:
1. Installs dependencies
2. Mounts Google Drive (dataset persists here across sessions)
3. Downloads THINGS-EEG via `osfclient`
4. Clones/updates the EEG-FineTune repo so `src/dataset.py` is importable
5. Runs sanity check — stats, NaN/Inf assert, 5-epoch plot
6. Builds CLIP embedding cache (once)

**Run cells top-to-bottom. Re-running is safe — downloads and caches are skipped if already present.**

## Step 1 — Install dependencies

In [ ]:
# Colab ships: torch, torchvision, numpy, scipy, matplotlib
!pip install -q "mne>=1.7.0" "open-clip-torch>=2.24.0" "osfclient>=0.0.12" "tqdm>=4.66.0"

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/EEG-FineTune')
DATA_ROOT    = PROJECT_ROOT / 'data' / 'things-eeg'
REPO_DIR     = Path('/content/EEG-FineTune')

DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DATA_ROOT    : {DATA_ROOT}')

## Step 3 — Clone / update the EEG-FineTune repo

In [ ]:
import subprocess, sys

# Replace with your GitHub repo URL after first push
REPO_URL = 'https://github.com/YOUR_USERNAME/EEG-FineTune.git'

if REPO_DIR.exists():
    print('Repo already cloned — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    print(f'Cloning {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready at', REPO_DIR)

## Step 4 — Download THINGS-EEG dataset

Hosted on OSF: https://osf.io/3jk45/ (~15-20 GB total).  
Files already on Drive are skipped automatically.

In [ ]:
import subprocess
from pathlib import Path

OSF_PROJECT_ID = '3jk45'
SUBJECTS = [f'{i:02d}' for i in range(1, 11)]

def osf_fetch(remote_path, local_path):
    local_path = Path(local_path)
    if local_path.exists():
        print(f'  [skip] {local_path.name}')
        return
    local_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'  Fetching {remote_path}...')
    result = subprocess.run(
        ['osf', '-p', OSF_PROJECT_ID, 'fetch', remote_path, str(local_path)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  [ERROR] {result.stderr.strip()}')
        print(f'  Manual download: https://osf.io/{OSF_PROJECT_ID}/files/')
        print(f'  Place file at: {local_path}')
    else:
        print(f'  [done] {local_path.name}')

print('Downloading image_metadata.npy...')
osf_fetch('osfstorage/image_metadata.npy', DATA_ROOT / 'image_metadata.npy')

print('\nDownloading EEG data per subject...')
for sub in SUBJECTS:
    for split in ['training', 'test']:
        fname = f'sub-{sub}_eeg_{split}.npy'
        osf_fetch(
            f'osfstorage/preprocessed_data/sub-{sub}/{fname}',
            DATA_ROOT / 'preprocessed_data' / f'sub-{sub}' / fname
        )

print('\nNote: stimulus image_set/ (~10 GB) is needed for CLIP cache (Step 6).')
print('Download separately from OSF if not already on Drive.')
print('\nDownload step complete.')

## Step 5 — Sanity check

In [ ]:
import numpy as np
import pprint

sub = '01'
eeg_path = DATA_ROOT / 'preprocessed_data' / f'sub-{sub}' / f'sub-{sub}_eeg_training.npy'

if not eeg_path.exists():
    raise FileNotFoundError(f'EEG file not found: {eeg_path}\nMake sure Step 4 completed successfully.')

eeg = np.load(eeg_path)  # (n_trials, n_channels, n_times)
n_trials, n_channels, n_times = eeg.shape
sfreq = 1000.0

print('=' * 50)
print('DATASET STATISTICS  sub-01 / training')
print('=' * 50)
pprint.pprint({
    'n_trials'          : n_trials,
    'n_channels'        : n_channels,
    'n_times (samples)' : n_times,
    'epoch_duration_ms' : n_times / sfreq * 1000,
    'sampling_rate_hz'  : sfreq,
    'dtype'             : str(eeg.dtype),
    'shape'             : eeg.shape,
    'value_range'       : f'[{eeg.min():.4f}, {eeg.max():.4f}]',
}, sort_dicts=False)

print()
has_nan = bool(np.any(np.isnan(eeg)))
has_inf = bool(np.any(np.isinf(eeg)))
if has_nan or has_inf:
    raise ValueError(f'Data quality check FAILED — NaN: {has_nan}, Inf: {has_inf}')
print('[PASS] No NaN or Inf values found in raw EEG data.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import bandpass_filter, zscore_normalize, crop_epoch

rng = np.random.default_rng(42)
trial_indices = rng.choice(n_trials, size=5, replace=False)

n_crop = int(500 * sfreq / 1000)  # samples in 500ms window
time_axis = np.linspace(0, 500, n_crop)

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
fig.suptitle(
    '5 Random EEG Epochs  (sub-01, bandpass 4-40Hz + z-score + 500ms crop)',
    fontsize=12
)

for ax, trial_idx in zip(axes, trial_indices):
    epoch = eeg[trial_idx].copy()
    epoch = bandpass_filter(epoch, sfreq)
    epoch = zscore_normalize(epoch)
    epoch = crop_epoch(epoch, sfreq, t_start=0.0, duration_ms=500.0)

    for ch in range(epoch.shape[0]):
        ax.plot(time_axis, epoch[ch], linewidth=0.4, alpha=0.5, color='steelblue')

    ax.set_ylabel(f'Trial {trial_idx}\n(z-score)', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylim(-5, 5)

axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()

out_path = PROJECT_ROOT / 'sanity_check_epochs.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Plot saved to {out_path}')
plt.show()

## Step 6 — Build CLIP embedding cache

Runs once (~10-20 min on T4). Saves `embeddings_cache.npy` to Drive.

In [ ]:
import torch
import numpy as np
from src.dataset import build_clip_embedding_cache

image_set_dir = DATA_ROOT / 'image_set'
cache_path    = DATA_ROOT / 'embeddings_cache.npy'

if not image_set_dir.exists():
    print(f'[SKIP] image_set/ not found at {image_set_dir}')
    print('Download stimulus images from OSF first, then re-run this cell.')
elif cache_path.exists():
    emb = np.load(cache_path)
    print(f'[SKIP] Cache already exists  shape={emb.shape}')
else:
    meta = np.load(DATA_ROOT / 'image_metadata.npy', allow_pickle=True).item()
    image_files = list(meta['image_path'])
    full_paths  = [str(image_set_dir / f) for f in image_files]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {device}')
    emb = build_clip_embedding_cache(full_paths, cache_path, device=device)
    print(f'Done. shape={emb.shape}')

## Phase 1 complete

If all cells above passed:
- Stats printed without errors
- `[PASS] No NaN or Inf values found` appeared
- Epoch plot saved to `MyDrive/EEG-FineTune/sanity_check_epochs.png`

Proceed to **Phase 2 — Signal Encoder**.